# CCA default - scib_sim1

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/cca_optuna/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_cca/optimize_cca_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "cca_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load CCA optimizer: {OPTIMIZER_PATH}")
cca_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = cca_opt
optimizer_spec.loader.exec_module(cca_opt)


In [2]:
DATASET_NAME = 'scib_sim1'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Batch'
LABEL_KEY = 'Group'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(10))
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_cca"

cca_opt.DATASET_NAME = DATASET_NAME
cca_opt.DATA_PATH = DATA_PATH
cca_opt.BATCH_KEY = BATCH_KEY
cca_opt.LABEL_KEY = LABEL_KEY
cca_opt.LAYER = LAYER
cca_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
cca_opt.USE_CASW = "CASW" in SELECTED_METRICS
cca_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
cca_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
cca_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
cca_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
cca_opt.USE_KBET = "KBET" in SELECTED_METRICS
cca_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'scib_sim1'
assert BATCH_KEY == 'Batch'
assert LABEL_KEY == 'Group'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(10))
assert cca_opt.DIMS_USE == 30
assert cca_opt.N_NEIGHBORS == N_NEIGHBORS
assert cca_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                              scib_sim1
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                Batch
label_key                                                Group
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
layer                                                   counts
Name: configuration, dtype: object

In [3]:
if not Path(cca_opt.R_SCRIPT_PATH).is_file():
    raise FileNotFoundError(cca_opt.R_SCRIPT_PATH)
if not Path(cca_opt.R_LIBS_PATH).is_dir():
    raise FileNotFoundError(cca_opt.R_LIBS_PATH)

cca_opt.ensure_balanced_clustering_importable()
cca_opt.validate_configuration(list(SELECTED_METRICS))
cca_opt.validate_dataset_contents()


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/pyximport/pyximport.py:51: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:671: ImportWarning: PyxLoader.exec_module() not found; falling back to load_module()


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group


Dataset preflight: n_obs=12097 n_batches=6 n_labels=7
ISOLISI preflight: min_batch_coverage=2 isolated_labels=['Group7']
CPCR preflight: eligible_labels=7


In [4]:
CCA_DEFAULT_ARGS = {
    "nfeatures": 2000,
    "npcs": 50,
    "dims_use": 30,
    "layer": "counts",
}


def run_cca_default(adata, seed):
    return cca_opt.run_cca(
        adata_obj=adata,
        nfeatures=CCA_DEFAULT_ARGS["nfeatures"],
        npcs=CCA_DEFAULT_ARGS["npcs"],
        dims_use=CCA_DEFAULT_ARGS["dims_use"],
        layer=CCA_DEFAULT_ARGS["layer"],
        seed=seed,
    )


pd.Series(CCA_DEFAULT_ARGS, name="default_setting")


nfeatures      2000
npcs             50
dims_use         30
layer        counts
Name: default_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return cca_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_cca_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


Cell type silhouette score...


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.932954149458342


optimised clustering against cca_metric_label
optimal cluster resolution: 0.8
optimal score: 0.9738951234819648
BRAS...


seed           9.000000
BAMI           0.973895
CASW           0.741422
SCGRAPHM       0.773298
ISOLISI        0.948498
BRAS           0.880541
KBET           0.098984
CPCR           0.968983
bio_score      0.859278
batch_score    0.649502
joint_score    0.775368
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
1,1,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
2,2,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
3,3,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
4,4,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
5,5,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
6,6,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
7,7,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
8,8,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368
9,9,0.973895,0.741422,0.773298,0.948498,0.880541,0.098984,0.968983,0.859278,0.649502,0.775368


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.973895,0.000000e+00,0.973895,0.973895
CASW,0.741422,0.000000e+00,0.741422,0.741422
SCGRAPHM,0.773298,0.000000e+00,0.773298,0.773298
ISOLISI,0.948498,1.110223e-16,0.948498,0.948498
BRAS,0.880541,0.000000e+00,0.880541,0.880541
KBET,0.098984,1.387779e-17,0.098984,0.098984
CPCR,0.968983,0.000000e+00,0.968983,0.968983
bio_score,0.859278,0.000000e+00,0.859278,0.859278
batch_score,0.649502,0.000000e+00,0.649502,0.649502
joint_score,0.775368,0.000000e+00,0.775368,0.775368
